# 这是整个项目核心的技术重点。
# 目标：在时间 t 计算的特征，只能用 t 及之前的数据；
# 标签用 t+1 的数据。

## 把清洗后的价格数据（prices.parquet）加工成带技术指标 + 预测标签的数据集（features.parquet），供后续机器学习使用。
## 1. 写 src/features/build_features.py
- 详见 /home/mlops/quant-mlops-project/src/features/build_features.py

In [ ]:
# 执行特征构建脚本，生成完整数据集
# 终端执行记录：
(quant-mlops-project) mlops@MS-CEVXSRKPSHOI:~/quant-mlops-project$ uv run python src/features/build_features.py
dropped 210 rows (warmup + label NaN), kept 29310
saved -> data/features/features.parquet
date range: 2015-02-02 ~ 2026-09-28
features: ['ret_1d', 'ret_5d', 'ret_20d', 'vol_5d', 'vol_20d', 'vol_ratio', 'price_pos', 'ma_ratio']

In [2]:
import pandas  as pd
df = pd.read_parquet('/home/mlops/quant-mlops-project/data/features/features.parquet')

In [3]:
df.describe()

,date,open,high,low,close,volume,ret_1d,ret_5d,ret_20d,vol_5d,vol_20d,vol_ma20,vol_ratio,high_20,low_20,price_pos,ma_5,ma_20,ma_ratio,label
count,29310,29310.000000,29310.000000,29310.000000,29310.000000,2.931000e+04,29310.000000,29310.000000,29310.000000,29310.000000,29310.000000,2.931000e+04,29310.000000,29310.000000,29310.000000,29310.000000,29310.000000,29310.000000,29310.000000,29310.000000
mean,2020-11-26 14:12:53.797000,167.076129,169.055528,165.067955,167.107074,8.250144e+07,0.001194,0.005932,0.023715,0.018035,0.019500,8.268391e+07,1.006427,176.760070,154.632257,0.595987,166.869968,165.994413,1.007366,0.001189
min,2015-02-02 00:00:00,0.462494,0.471493,0.453397,0.466182,7.140000e+05,-0.263901,-0.430459,-0.606265,0.000687,0.003487,1.852515e+06,0.195451,0.490324,0.453397,0.000000,0.466994,0.474475,0.668661,-0.263901
25%,2017-12-27 00:00:00,55.574788,56.199215,55.028860,55.593318,1.172828e+07,-0.008500,-0.017894,-0.028045,0.009390,0.011970,1.245942e+07,0.762193,58.270024,50.898333,0.348632,55.486705,54.713302,0.987160,-0.008500
50%,2020-11-24 00:00:00,135.245602,136.665487,133.740979,135.254555,2.997065e+07,0.001109,0.005768,0.021914,0.014372,0.016447,3.161166e+07,0.916997,142.438116,125.777513,0.651614,135.275327,134.575531,1.008360,0.001103
75%,2023-10-25 00:00:00,231.996629,234.570007,229.369325,232.197712,8.009350e+07,0.010981,0.028649,0.068313,0.022132,0.023541,8.385087e+07,1.132896,244.421734,214.322269,0.861977,231.595573,230.782082,1.028044,0.010974
max,2026-09-28 00:00:00,787.943661,793.022965,777.655507,786.798218,3.692928e+09,0.298068,0.564756,1.063036,0.147880,0.100268,1.127078e+09,6.691014,793.022965,728.037428,1.000000,777.796899,761.150146,1.286188,0.298068
std,NaN,139.339503,140.991459,137.634677,139.320507,1.495164e+08,0.022458,0.049011,0.098835,0.013701,0.011301,1.361562e+08,0.408674,148.093364,129.274890,0.295333,139.136284,138.537019,0.041537,0.022456


In [4]:
df.head()

,date,ticker,open,high,low,close,volume,ret_1d,ret_5d,ret_20d,...,vol_20d,vol_ma20,vol_ratio,high_20,low_20,price_pos,ma_5,ma_20,ma_ratio,label
0,2015-02-02,AAPL,26.099663,26.347283,25.664116,26.227894,250956400,0.012547,0.048895,0.085064,...,0.023813,262959580.0,0.954354,26.530790,23.132634,0.910865,25.608399,24.564523,1.042495,0.000169
1,2015-02-03,AAPL,26.199153,26.329595,26.002383,26.232317,207662800,0.000169,0.087136,0.116706,...,0.022587,260485620.0,0.797214,26.530790,23.132634,0.912166,26.028913,24.701599,1.053734,0.007669
2,2015-02-04,AAPL,26.199149,26.643540,26.157142,26.433504,280598800,0.007669,0.036857,0.125165,...,0.022551,261356140.0,1.073626,26.643540,23.258658,0.937949,26.216840,24.848624,1.055062,0.007137
3,2015-02-05,AAPL,26.639927,26.686541,26.469017,26.622171,168984800,0.007137,0.012728,0.117525,...,0.022477,261784200.0,0.645512,26.686541,23.258658,0.981222,26.283756,24.988610,1.051829,-0.008421
4,2015-02-06,AAPL,26.639935,26.690987,26.291454,26.397997,174826400,-0.008421,0.019114,0.067114,...,0.021310,258652620.0,0.675912,26.690987,23.258658,0.914638,26.382777,25.071623,1.052296,0.006642


In [5]:
df.tail()

,date,ticker,open,high,low,close,volume,ret_1d,ret_5d,ret_20d,...,vol_20d,vol_ma20,vol_ratio,high_20,low_20,price_pos,ma_5,ma_20,ma_ratio,label
29305,2026-09-22,V,371.760010,372.000000,359.799988,362.040009,8773500,-0.021381,-0.036154,-0.053267,...,0.010045,6307250.0,1.391018,385.570007,359.799988,0.086924,368.228003,373.964503,0.984660,-0.001436
29306,2026-09-23,V,361.239990,366.329987,360.000000,361.519989,5125600,-0.001436,-0.025369,-0.058885,...,0.009907,6224650.0,0.823436,385.570007,359.799988,0.066744,366.346002,372.833502,0.982599,0.017869
29307,2026-09-24,V,362.269989,368.209991,362.000000,367.980011,4415500,0.017869,-0.005271,-0.041469,...,0.010947,6141670.0,0.718941,383.679993,359.799988,0.342547,365.956006,372.037503,0.983654,-0.001631
29308,2026-09-25,V,366.019989,368.329987,363.790009,367.380005,4576700,-0.001631,-0.002471,-0.032345,...,0.010741,6038430.0,0.757929,383.679993,359.799988,0.317421,365.774005,371.423503,0.984790,0.000980
29309,2026-09-28,V,366.700012,370.500000,366.059998,367.739990,6381000,0.000980,-0.005974,-0.036321,...,0.010645,6125330.0,1.041740,382.480011,359.799988,0.350088,365.332001,370.730502,0.985438,-0.005058


In [ ]:
# 抽查 AAPL 的前 25 行特征数据，目的是验证特征计算是否正确、有没有未来函数。
# 终端执行记录如下：
(quant-mlops-project) mlops@MS-CEVXSRKPSHOI:~/quant-mlops-project$ uv run python -c "
import pandas as pd
df = pd.read_parquet('data/features/features.parquet')
a = df[df['ticker']=='AAPL'].sort_values('date').head(25).reset_index(drop=True)
pd.set_option('display.width', 200)
print(a[['date','close','ret_1d','vol_5d','label']].to_string())
"
         date      close    ret_1d    vol_5d     label
0  2015-02-02  26.227894  0.012547  0.036233  0.000169
1  2015-02-03  26.232317  0.000169  0.027687  0.007669
2  2015-02-04  26.433504  0.007669  0.016790  0.007137
3  2015-02-05  26.622171  0.007137  0.010585 -0.008421
4  2015-02-06  26.397997 -0.008421  0.008142  0.006642
5  2015-02-09  26.573341  0.006642  0.006892  0.019212
6  2015-02-10  27.083858  0.019212  0.009823  0.023439
7  2015-02-11  27.718668  0.023439  0.012484  0.012652
8  2015-02-12  28.069372  0.012652  0.012456  0.004903
9  2015-02-13  28.206989  0.004903  0.007953  0.005901
10 2015-02-17  28.373451  0.005901  0.008115  0.006963
11 2015-02-18  28.571011  0.006963  0.007692 -0.002098
12 2015-02-19  28.511076 -0.002098  0.005277  0.008174
13 2015-02-20  28.744129  0.008174  0.004027  0.027028
14 2015-02-23  29.521011  0.027028  0.010752 -0.006241
15 2015-02-24  29.336777 -0.006241  0.012848 -0.025573
16 2015-02-25  28.586548 -0.025573  0.019328  0.012656
17 2015-02-26  28.948339  0.012656  0.019993 -0.015028
18 2015-02-27  28.513294 -0.015028  0.021208  0.004904
19 2015-03-02  28.653130  0.004904  0.015260  0.002091
20 2015-03-03  28.713053  0.002091  0.015658 -0.006339
21 2015-03-04  28.531042 -0.006339  0.010654 -0.016570
22 2015-03-05  28.058275 -0.016570  0.009715  0.001503
23 2015-03-06  28.100443  0.001503  0.008716  0.004265
24 2015-03-09  28.220301  0.004265  0.008577 -0.020686

### 验证逻辑：
- 第 t 行的 ret_1d = (close[t] / close[t-1]) - 1
- 第 t 行的 label = (close[t+1] / close[t]) - 1
- label 和下一行的 ret_1d 应该相等（因为 label 是“明天的收益”，下一行的 ret_1d 就是“明天相对今天的收益”）
- 如果对得上，point-in-time 就是对的。